In [1]:
pip install torchcodec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 38.3 MB/s eta 0:00:00


In [2]:
import librosa
import soundfile as sf
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import numpy as np
import matplotlib.pyplot as plt
import torchaudio
import torchvision
import pandas as pd
import os
import math
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
from tqdm import tqdm
from sklearn.metrics import f1_score

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **Dataset for window-wise onset/offset detector**



In [ ]:
# functions
def parse_exemplar_name(fname):
  base = os.path.splitext(fname)[0]
  _, pitch_str, vel_str, tempo_str = base.split('_')
  vel = int(vel_str)
  tempo = int(tempo_str)
  midi = librosa.note_to_midi(pitch_str)
  return midi, vel, tempo

def BuildNoteEvents(onset, velocity, max_notes=10, n_pitches=84):
  T, P = velocity.shape
  note_events = np.zeros((T, max_notes, 3), dtype=np.float32)
  last_onset_frame = np.full(P, -1, dtype=np.int32)

  for t in range(T):
    rows = []
    for p in range(P):
      if onset[t,p] > 0:
        last_onset_frame[p] = t

      if velocity[t,p] > 0:
        vel = float(velocity[t,p])
        if last_onset_frame[p] < 0:
          dur = 0
        else:
          dur = t - last_onset_frame[p]
        rows.append([p, vel, dur])

    rows.sort(key=lambda x:x[0])

    if len(rows) > max_notes:
      rows = rows[:max_notes]
    else:
      rows += ([[n_pitches, 0, 0]] * (max_notes - len(rows)))

    note_events[t] = np.array(rows, dtype=np.float32)

  return note_events

def build_vel_target(vel_rows, onset_rows, n_pitches=84):
  T = vel_rows.shape[0]
  target = np.zeros((T, n_pitches), dtype=np.float32)
  for i in range(vel_rows.shape[0]):
    vel_row = vel_rows[i]
    onset_row = onset_rows[i]
    pitches = np.where(onset_row > 0)[0]
    target[i,pitches] = vel_rows[i,pitches] / 127
  return target

def full_window(full, t, win, bilateral=True):
  start = max(0, t - win)
  if bilateral == False:
    win_len = win
    end = t
    X_win = full[:,start:end,:]
    if X_win.shape[1] < win_len:
      pad = win_len - X_win.shape[1]
      X_win = torch.nn.functional.pad(X_win, (0,0,pad,0,0,0))
  else:
    win_len = win * 2
    end = min(t + win, full.shape[1])
    X_win = full[:,start:end,:]
    if X_win.shape[1] < win_len:
      pad = win_len - X_win.shape[1]
      if t < win:
        X_win = torch.nn.functional.pad(X_win, (0,0,pad,0,0,0))
      elif t >= full.shape[1] - win:
        X_win = torch.nn.functional.pad(X_win, (0,0,0,pad,0,0))

  return X_win

def compute_hcqt(y, sr, harmonics=(0.5,1,2,3,4), fmin=librosa.note_to_hz("C1"),
                 bins_per_octave=12, n_bins=84):
    hcqt_list = []
    for h in harmonics:
        cqt = librosa.cqt(y, sr=sr, fmin=fmin*h, n_bins=n_bins,
                          bins_per_octave=bins_per_octave)
        hcqt_list.append(np.abs(cqt))
    hcqt = np.stack(hcqt_list, axis=0)  # (H, F, T)
    hcqt /= hcqt.max(axis=(-1, -2), keepdims=True)
    return hcqt

# the dataset
class WOODDataset(Dataset):
  def __init__(self, root, n_events=10, win_sec=1, hop=512, n_pitches=84, sr=44100):
    self.root = root
    self.n_events = n_events
    self.hop = hop
    self.win_sec = win_sec
    self.sr = sr
    self.n_pitches = n_pitches

    self.win = math.ceil(win_sec * sr / hop) # number of frames

    self.folders = sorted([
        f for f in os.listdir(root) if os.path.isdir(os.path.join(root, f))
    ])

    self.folder_data = []
    self.index = []

    print("Loading all folders...")
    for f_idx, folder in enumerate(self.folders):
      fd = self.load_folder(os.path.join(root, folder), n_pitches)
      T = fd["frame"].shape[0]

      for t in range(1, T, self.win):
        self.index.append((f_idx, t))

      self.folder_data.append(fd)

    print(f"Total windows for training: {len(self.index)}")

  def load_folder(self, folder_path, n_pitches=84):
    exemplar_path = None
    for f in os.listdir(folder_path):
      if f.startswith("exemplar") and f.endswith(".wav"):
        exemplar_path = os.path.join(folder_path,f)
        break

    midi_ex, vel_ex, tempo_ex = parse_exemplar_name(os.path.basename(exemplar_path))

    # exemplar audio
    y_ex, _ = librosa.load(exemplar_path, sr=self.sr)
    if y_ex.shape[0] < self.sr:
      y_ex = np.pad(y_ex, (0, self.sr - y_ex.shape[0]))
    else:
      y_ex = y_ex[:self.sr]
    S_ex = compute_hcqt(y_ex, self.sr)
    S_ex = librosa.amplitude_to_db(np.abs(S_ex))
    S_ex = np.transpose(S_ex, (0, 2, 1))

    # full audio
    y, _ = librosa.load(os.path.join(folder_path, "full.wav"), sr=self.sr)
    S = compute_hcqt(y, self.sr)
    S = librosa.amplitude_to_db(np.abs(S))
    S = np.transpose(S, (0, 2, 1))

    Ts = S.shape[1] # total number of frames per song

    # .csv events
    df = pd.read_csv(os.path.join(folder_path,"midi_export.csv"))

    onset = np.zeros((Ts,n_pitches), np.float32)
    offset = np.zeros((Ts,n_pitches), np.float32)
    frame = np.zeros((Ts,n_pitches), np.float32)
    velocity = np.zeros((Ts,n_pitches), np.float32)

    for _, row in df.iterrows():
      sf = int(row.start_sec * self.sr / self.hop)
      ef = int(row.end_sec * self.sr / self.hop)
      p  = int(row.pitch - 24)
      if 0 <= p < n_pitches:
        onset[sf,p]  = 1.0
        if ef < Ts:
          offset[ef,p] = 1.0
        frame[sf:ef,p] = 1.0
        velocity[sf:ef,p] = row.velocity

    # note event tensor
    note_events = BuildNoteEvents(onset, velocity, max_notes=self.n_events)

    return dict(
        S_full=torch.tensor(S, dtype=torch.float32),
        onset=torch.tensor(onset, dtype=torch.float32),
        offset=torch.tensor(offset, dtype=torch.float32),
        frame=torch.tensor(frame, dtype=torch.float32),
        velocity=torch.tensor(velocity, dtype=torch.float32),
        note_events=torch.tensor(note_events, dtype=torch.float32),
        S_ex=torch.tensor(S_ex, dtype=torch.float32),
        midi_ex=midi_ex,
        vel_ex=vel_ex
    )

  def __len__(self):
    return len(self.index)

  def __getitem__(self, idx):
    f_idx, t = self.index[idx]
    fd = self.folder_data[f_idx]

    S_full = fd["S_full"]   # (Ts,n_pitches)
    note_ev = fd["note_events"]  # (Ts,n,3)
    frame = fd["frame"]
    onset = fd["onset"]
    offset = fd["offset"]
    velocity = fd["velocity"]

    # full window of shape (win, n_pitches)
    X_win = full_window(S_full, t, self.win) # (Tw,n_pitches)

    prev_frame = frame[t-self.win:t]
    prev_vel = velocity[t-self.win:t]
    if t > 1:
      prev_ev = note_ev[t-self.win:t]
    else:
      prev_ev = torch.zeros((self.win, self.n_events, 3), dtype=torch.float32)
      prev_ev[:,:,0] = float(n_pitches)

    offset_prev_ev = note_ev[t-1]
    onset_target = onset[t:t+self.win]
    offset_target = offset[t:t+self.win]
    vel_target = build_vel_target(velocity[t:t+self.win], onset[t:t+self.win])
    vel_target = torch.tensor(vel_target, dtype=torch.float32)

    Tw = S_full.shape[1] # total number of frames per window
    if Tw - t < self.win:
      onset_target = F.pad(onset_target, (0,0,0,self.win - Tw + t))
      offset_target = F.pad(offset_target, (0,0,0,self.win - Tw + t))
      vel_target = F.pad(vel_target, (0,0,0,self.win - Tw + t))

    return {
      "X_full_win": X_win,           # (win*2,h,n_pitches)
      "X_ex": fd["S_ex"],            # (win,h,n_pitches)
      "note_prev": prev_ev,          # (win,n,3)
      "onset_target": onset_target,  # (win,n_pitches)
      "offset_target": offset_target, # (win,n_pitches)
      "vel_target": torch.tensor(vel_target, dtype=torch.float32), # (win,n_pitches)
      "pitch_ex": fd["midi_ex"],
      "vel_ex":   fd["vel_ex"]
    }

dataset = WOODDataset("/content/drive/MyDrive/MIDI/train") # to be modified for real training data directory
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

# **Window-wise onset/offset detection model**

In [14]:
def crop_to_match(enc, dec):
    '''
    Center-crop enc to match dec's time/freq dims.
    '''
    _,_,Te,Fe = enc.shape
    _,_,Td,Fd = dec.shape
    te = (Te - Td) // 2
    fe = (Fe - Fd) // 2
    return enc[:, :, te:te+Td, fe:fe+Fd]

class UNetMPE(nn.Module):
  def __init__(self, freq_bins=84, pitches=128, h=5):
    super().__init__()
    def conv_block(in_c, out_c):
      return nn.Sequential(
        nn.Conv2d(in_c, out_c, 3, padding=1),
        nn.BatchNorm2d(out_c),
        nn.ReLU(),
        nn.Conv2d(out_c, out_c, 3, padding=1),
        nn.BatchNorm2d(out_c),
        nn.ReLU()
      )

    self.enc1 = conv_block(h, 16)
    self.enc2 = conv_block(16, 32)
    self.enc3 = conv_block(32, 64)

    self.pool = nn.MaxPool2d((2,2))
    self.bottleneck = conv_block(64, 128)

    self.up3 = nn.ConvTranspose2d(128, 64, 2, stride=2)
    self.dec3 = conv_block(128, 64)

    self.up2 = nn.ConvTranspose2d(64, 32, 2, stride=2)
    self.dec2 = conv_block(64, 32)

    self.up1 = nn.ConvTranspose2d(32, 16, 2, stride=2)
    self.dec1 = conv_block(32, 16)

    self.out_conv = nn.Conv2d(16, pitches, kernel_size=1)

  def forward(self, x):
    e1 = self.enc1(x)
    p1 = self.pool(e1)

    e2 = self.enc2(p1)
    p2 = self.pool(e2)

    e3 = self.enc3(p2)
    p3 = self.pool(e3)

    b = self.bottleneck(p3)

    u3 = self.up3(b)
    e3_cropped = crop_to_match(e3, u3)
    u3 = torch.cat([u3, e3_cropped], dim=1)
    d3 = self.dec3(u3)

    u2 = self.up2(d3)
    e2_cropped = crop_to_match(e2, u2)
    u2 = torch.cat([u2, e2_cropped], dim=1)
    d2 = self.dec2(u2)

    u1 = self.up1(d2)
    e1_cropped = crop_to_match(e1, u1)
    u1 = torch.cat([u1, e1_cropped], dim=1)
    d1 = self.dec1(u1)

    out = self.out_conv(d1)
    out = out.mean(dim=-1)
    out = out.transpose(1,2)
    return out

class WOODModel(nn.Module):
  def __init__(self, n_events=7, n_pitches=84, d_model=256, h=5):
    super().__init__()
    self.n = n_events
    self.h = h

    # windowed HCQT encoder
    self.win_conv = nn.Sequential(
      nn.Conv2d(h, 32, (3,3), padding=(1,1)),
      nn.ReLU(),
      nn.Conv2d(32, 64, (3,3), padding=(1,1)),
      nn.ReLU(),
      nn.Conv2d(64, d_model, (3,3), padding=(1,1)),
      nn.ReLU(),
    )

    # windowed HCQT salience encoder
    self.salience_net = UNetMPE(freq_bins=n_pitches, pitches=n_pitches)
    self.salience_proj = nn.Linear(n_pitches, d_model)

    # exemplar HCQT encoders
    self.ex_conv = nn.Sequential(
      nn.Conv2d(h, 32, (3,3), padding=(1,1)),
      nn.ReLU(),
      nn.Conv2d(32, 64, (3,3), padding=(1,1)),
      nn.ReLU(),
      nn.Conv2d(64, d_model, (3,3), padding=(1,1)),
      nn.ReLU(),
    )
    self.ex_pool = nn.AdaptiveAvgPool2d((1,1))
    self.ex_pitch_emb = nn.Embedding(129, d_model)
    self.ex_vel_proj = nn.Linear(1, d_model)

    # note event encoders
    self.pitch_emb = nn.Embedding(NUM_PITCH_INDICES:=129, d_model)
    self.vel_emb = nn.Embedding(NUM_PITCH_INDICES:=129, d_model)

    self.dur_mlp = nn.Sequential(
      nn.Linear(1, d_model),
      nn.ReLU(),
      nn.Linear(d_model, d_model)
    )

    self.row_mlp = nn.Sequential(
      nn.Linear(d_model*3, d_model),
      nn.ReLU(),
      nn.Linear(d_model, d_model)
    )
    self.prev_gate = nn.Parameter(torch.tensor(0.5))

    # fuse embeddings
    self.fusion_mlp = nn.Sequential(
      nn.Linear(d_model*4, d_model),
      nn.ReLU(),
      nn.Linear(d_model, d_model)
    )

    # temporal model
    self.gru = nn.GRU(d_model, d_model, batch_first=True, bidirectional=False)

    # output heads
    self.onset_head = nn.Linear(d_model, n_pitches)
    self.offset_head = nn.Linear(d_model, n_pitches)
    self.vel_head    = nn.Linear(d_model, n_pitches)

  def forward(self, full_win, ex, ex_pitch, ex_vel, prev_state):
    """
    full_win: (B, Tw, n_mel)
    ex: (B, Te, n_mel)
    prev_state: (B, n, 3)
    """

    B = full_win.shape[0]

    # window encoding
    w = full_win.transpose(2,3)
    w_h = self.win_conv(w)
    w_h = w_h.mean(2).transpose(1,2) # (B,Tw,d)
    Tw = w_h.size(1) # total window length
    hw = Tw // 2  # half window length

    # salience U-Net model
    sal = self.salience_net(full_win) # (B, T_sal, n_pitches)
    sal = F.avg_pool1d(sal.transpose(1,2), kernel_size=2, stride=2).transpose(1,2)
    sal_h = self.salience_proj(sal)  # (B, T_sal, d_model)
    T_target = Tw
    T_sal = sal_h.size(1) # T_sal not necessarily the same as Tw
    pad = T_target - T_sal      # 6
    pad_left = pad // 2         # symmetric padding: 3 left, 3 right
    pad_right = pad - pad_left
    sal_h = F.pad(sal_h, (0, 0, pad_left, pad_right))   # (B,Tw,256)

    # exemplar embeddings
    x = ex.transpose(2,3)
    ex_h = self.ex_conv(ex)
    ex_h = self.ex_pool(ex_h).squeeze(-1).squeeze(-1) # (B,d)

    ex_pitch_emb = self.ex_pitch_emb(ex_pitch) # (B,d)
    ex_vel_emb   = self.ex_vel_proj(ex_vel)
    ex_all = ex_h + ex_pitch_emb + ex_vel_emb

    # prev_state encoding
    prev_state = prev_state[:,:,:self.n,:]
    pitch_idx = prev_state[:,:,:,0].long() # (B,Tw,n)
    vel_idx = prev_state[:,:,:,1].long() # (B,Tw,n)
    dur = prev_state[:,:,:,2:3].float() # (B,Tw,n,1)

    pitch_vec = self.pitch_emb(pitch_idx) # (B,n,d)
    vel_vec = self.vel_emb(vel_idx) # (B,n,d)
    dur_vec = self.dur_mlp(dur) # (B,n,d)

    row_vec = self.row_mlp(torch.cat([pitch_vec, vel_vec, dur_vec], dim=-1))
    rows_agg = row_vec.mean(dim=(1,2)) # (B,d)
    rows_agg = self.prev_gate * rows_agg

    # fusion per time frame
    ex_b = ex_all.unsqueeze(1).expand(-1, Tw, -1) # (B,Tw,d)
    rows_b = rows_agg.unsqueeze(1).expand(-1, Tw, -1) # (B,Tw,d)
    fused = self.fusion_mlp(torch.cat([w_h, sal_h, ex_b, rows_b], dim=-1)) # (B,Tw,d)

    # temporal model
    out_seq, _ = self.gru(fused)
    out_seq = out_seq[:, hw:, :] # only take a size of number of current frames

    # heads
    onset  = self.onset_head(out_seq)
    offset = self.offset_head(out_seq)
    vel = torch.sigmoid(self.vel_head(out_seq))

    return onset, offset, vel


# **Training loop**

In [ ]:
n = 7
model = WOODModel(n_events=n).to(device)
criterion = nn.BCEWithLogitsLoss()

criterion_on = nn.BCELoss()
criterion_off = nn.BCELoss()
criterion_vel = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
lam = 0.5 # velocity weight

offset_arrays = []
onset_arrays = []
offset_gts = []
onset_gts = []
epoch_onset_losses = []
epoch_offset_losses = []
epoch_vel_losses = []
epoch_losses = []

for epoch in range(200):
    model.train()
    batch_idx = 0
    losses = []
    offset_losses = []
    onset_losses = []
    vel_losses = []

    for batch in dataloader:
        full_win   = batch["X_full_win"].to(device)    # (B,Tw,n_pitches)
        ex_win     = batch["X_ex"].to(device)      # (B,Te,n_pitches)
        prev_note  = batch["note_prev"].to(device)     # (B,T,11,3)
        onset_target = batch["onset_target"].to(device)   # (B,T,n_pitches)
        offset_target = batch["offset_target"].to(device)  # (B,T,n_pitches)
        vel_target = batch["vel_target"].to(device)     # (B,T,n_pitches)
        pitch_ex = batch["pitch_ex"].to(device)
        vel_ex = batch["vel_ex"].to(device)

        prev_note = prev_note[:,:n,:] # (B,T,n,3)
        vel_ex = vel_ex.float() / 127.0
        vel_ex = vel_ex.unsqueeze(-1)

        # Forward
        onset_pred, offset_pred, vel_pred = model(full_win, ex_win, pitch_ex, vel_ex, prev_note)

        L_on = criterion(onset_pred, onset_target)
        L_off = criterion(offset_pred, offset_target)
        L_vel = (torch.abs(vel_pred - vel_target) * onset_target).sum() \
              / (onset_target.sum() + 1e-6)   # only where onset exists

        loss = L_on + L_off + lam * L_vel

        # Backprop
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_idx += 1
        losses.append(loss.item())
        offset_losses.append(L_off.item())
        onset_losses.append(L_on.item())
        vel_losses.append(L_vel.item())

    loss_avg = np.mean(losses)
    offset_loss_avg = np.mean(offset_losses)
    onset_loss_avg = np.mean(onset_losses)
    vel_loss_avg = np.mean(vel_losses)
    epoch_losses.append(loss_avg)
    epoch_offset_losses.append(offset_loss_avg)
    epoch_onset_losses.append(onset_loss_avg)
    epoch_vel_losses.append(vel_loss_avg)

    print(f"Epoch:{epoch+1},Loss:{loss_avg:.4f},Offset Loss:{offset_loss_avg:.4f},Onset Loss:{onset_loss_avg:.4f},Vel Loss:{vel_loss_avg:.4f}")

torch.save(model.state_dict(), "mpe_exemplar_wood.pth")

# plot training losses
epochs = range(1, len(epoch_losses) + 1)
plt.figure(figsize=(12,4))
plt.subplot(1,2,1)
plt.plot(epochs[3:], np.array(epoch_losses)[3:], label="Total Loss")
plt.title("Total Training Loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.subplot(1,2,2)
plt.plot(epochs[3:], np.array(epoch_offset_losses)[3:], 'r', label="Offset Loss")
plt.plot(epochs[3:], np.array(epoch_onset_losses)[3:], 'g', label="Onset Loss")
plt.plot(epochs[3:], np.array(epoch_vel_losses)[3:], 'b', label="Velocity Loss")
plt.legend()
plt.title("Training Losses")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.show()

# **Test on the trained model**

In [ ]:
class WOODTestDataset(Dataset):
    def __init__(self, root, hop=512, n_events=10, n_pitches=84, sr=44100):
        self.root = root
        self.hop = hop
        self.n_events = n_events
        self.sr = sr
        self.n_pitches = n_pitches

        self.folders = sorted([
          f for f in os.listdir(root) if os.path.isdir(os.path.join(root, f))
        ])

    def __len__(self):
        return len(self.folders)

    def __getitem__(self, idx):
        folder = self.folders[idx]
        folder_path = os.path.join(self.root, folder)

        for f in os.listdir(folder_path):
            if f.startswith("exemplar") and f.endswith(".wav"):
                exemplar_path = os.path.join(folder_path, f)
                break

        pitch_ex, vel_ex, tempo_ex = parse_exemplar_name(os.path.basename(exemplar_path))

        # exemplar audio
        y_ex, _ = librosa.load(exemplar_path, sr=self.sr)
        if y_ex.shape[0] < self.sr:
          y_ex = np.pad(y_ex, (0, self.sr - y_ex.shape[0]))
        else:
          y_ex = y_ex[:self.sr]
        S_ex = compute_hcqt(y_ex, self.sr)
        S_ex = librosa.amplitude_to_db(np.abs(S_ex))
        S_ex = np.transpose(S_ex, (0, 2, 1))

        # full audio
        y, _ = librosa.load(os.path.join(folder_path, "full.wav"), sr=self.sr)
        S = compute_hcqt(y, self.sr)
        S = librosa.amplitude_to_db(np.abs(S))
        S_full = np.transpose(S, (0, 2, 1))

        T = S_full.shape[1]

        # Load GT MIDI sheet
        csv_path = os.path.join(folder_path, "midi_export.csv")
        df = pd.read_csv(csv_path)

        onset = np.zeros((T, self.n_pitches), dtype=np.float32)
        offset = np.zeros((T, self.n_pitches), dtype=np.float32)
        frame = np.zeros((T, self.n_pitches), dtype=np.float32)
        velocity = np.zeros((T, self.n_pitches), dtype=np.float32)

        for _, row in df.iterrows():
            p = int(row.pitch - 24) # discard lowest 24 pitches
            if p < 0 or p >= self.n_pitches:
                continue
            sf = int(row.start_sec * self.sr / self.hop)
            ef = int(row.end_sec   * self.sr / self.hop)
            onset[sf, p] = 1
            if ef < T:
              offset[ef, p] = 1
            frame[sf:ef, p] = 1
            velocity[sf:ef, p] = row.velocity

        # Full note-event tensor
        note_events = BuildNoteEvents(onset, velocity, max_notes=self.n_events) # (T,n,3)

        return {
            "song_id": folder,
            "X_full": torch.tensor(S_full, dtype=torch.float32),
            "X_ex": torch.tensor(S_ex, dtype=torch.float32),
            "pitch_ex": pitch_ex,
            "vel_ex": vel_ex,
            "onset": torch.tensor(onset, dtype=torch.float32),
            "offset": torch.tensor(offset, dtype=torch.float32),
            "frame": torch.tensor(frame, dtype=torch.float32),
            "velocity": torch.tensor(velocity, dtype=torch.float32),
            "note_events": torch.tensor(note_events, dtype=torch.float32)
        }

testset = WOODTestDataset("/content/drive/MyDrive/MIDI/test") # to be modified for real testing data directory
testloader = DataLoader(testset, batch_size=1, shuffle=False)
print("Number of songs read:", len(testset))

In [9]:
def update_note_event(prev_note_frame, onset_bin, offset_bin, vel_value, n_pitches=84):
    """
    prev_note_frame: (1, n, 3)
    onset_bin:       (1, T, n_pitches)
    offset_bin:      (1, T, n_pitches)
    vel_value:       (1, T, n_pitches)
    return:          (1, T, n, 3)
    n_pitches is also the value indicating empty pitch
    """

    B, n, _ = prev_note_frame.shape
    T = onset_bin.shape[1]
    device = prev_note_frame.device

    # Initialize output
    out = torch.zeros((B, T, n, 3), device=device)
    cur = prev_note_frame.clone()  # (1, n, 3)

    for t in range(T):
        onset_t  = onset_bin[:, t, :]   # (1, n_pitches)
        offset_t = offset_bin[:, t, :]   # (1, n_pitches)
        vel_t    = vel_value[:, t, :]   # (1, n_pitches)

        # Apply offsets
        for p in torch.nonzero(offset_t[0], as_tuple=False).flatten():
            mask = cur[0, :, 0] == p
            cur[0, mask] = torch.tensor(
                [n_pitches, 0.0, 0.0], device=device
            )

        # Increment duration
        active_mask = cur[0, :, 0] != n_pitches
        cur[0, active_mask, 2] += 1.0

        # Apply onsets
        for p in torch.nonzero(onset_t[0], as_tuple=False).flatten():
            active = (cur[0, :, 0] == p).any() # Check if pitch already active
            if not active:
                empty_rows = (cur[0, :, 0] == n_pitches).nonzero(as_tuple=False)
                if len(empty_rows) > 0:
                    r = empty_rows[0, 0]
                    cur[0, r, 0] = p
                    cur[0, r, 1] = vel_t[0, p]
                    cur[0, r, 2] = 0.0  # duration starts at 0

        # Sort rows by pitch
        pitches = cur[0, :, 0]
        sort_key = pitches.clone()
        sort_key[sort_key == n_pitches] = 1e9  # push empty rows to end
        idx = torch.argsort(sort_key)
        cur = cur[:, idx, :]

        # Store
        out[:, t, :, :] = cur
    return out

def reconstruct_frame_matrix(note_event_windows, n_pitches=84, vel=True):
    """
    note_event_windows: list of tensors, each (1, Ts, n, 3)
    return: frame matrix (Ts, n_pitches)
    vel = True or False: whether to return velocity or frame matrix
    """
    frames = []
    for win in note_event_windows:
        win = win.squeeze(0)  # (Ts, n, 3)
        for t in range(win.shape[0]):
            frame = torch.zeros(n_pitches, dtype=torch.float32)
            events = win[t]  # (n, 3)
            for row in events:
                pitch = int(row[0])
                if 0 <= pitch < n_pitches:
                  if vel == True:
                    frame[pitch] = row[1] # velocity
                  else:
                    frame[pitch] = 1.0
            frames.append(frame)
    return torch.stack(frames).int()

def reconstruct_onset_matrix(note_event_windows, n_pitches=84):
    """
    Onset occurs when pitch appears that was not active in previous frame
    """
    onsets = []
    for win in note_event_windows:
        win = win.squeeze(0)  # (Ts, n, 3)
        for t in range(win.shape[0]):
            onset = torch.zeros(n_pitches, dtype=torch.bool)
            for row in win[t]:
                pitch = int(row[0])
                dur = int(row[2])
                if 0 <= pitch < n_pitches and dur == 0:
                    onset[pitch] = 1.0
            onsets.append(onset.float())
    return torch.stack(onsets).int()


# **Testing loop**

In [ ]:
n = 7
sr = 44100
hop = 512
n_pitches = 84
window = math.ceil(sr/hop)
model = WOODModel(n_events=n).to(device)
onset_th = 0.002
offset_th = 0.001
bilateral = True

with torch.no_grad():
  model.load_state_dict(torch.load("mpe_exemplar_wood.pth"))

model.eval()

test_song = testset[1]
full_mel = test_song["X_full"].to(device)
ex_mel = test_song["X_ex"].to(device)
pitch_ex = torch.tensor(test_song["pitch_ex"]).unsqueeze(0).to(device)
vel_ex = torch.tensor(test_song["vel_ex"]).unsqueeze(0).to(device)
onset_gt = test_song["onset"].to("cpu")
offset_gt = test_song["offset"].to("cpu")
frame_gt = test_song["frame"].to("cpu")
vel_gt = test_song["velocity"].to("cpu")

Ts = full_mel.shape[1] # number of frames in this song

ex_win   = ex_mel.unsqueeze(0).to(device)     # (1,T,n_pitches)
prev_note = torch.zeros((window, n, 3), dtype=torch.float32)
prev_note[:,:,0] = float(n_pitches)
prev_note = prev_note.unsqueeze(0).to(device)
vel_ex = vel_ex.float() / 127.0
vel_ex = vel_ex.unsqueeze(-1)

results_onsets = []
results_offsets = []
results_notes = []

for t in range(0, Ts, window):
  # Extract windows
  full_win = full_window(full_mel, t, window, bilateral=True)
  full_win = full_win.unsqueeze(0).to(device)   # (1,Tw,n_pitches)

  # Model inference
  with torch.no_grad():
    onset_pred, offset_pred, vel_pred = model(full_win, ex_win, pitch_ex, vel_ex, prev_note)

  # Post-process logits
  onset_bin = (torch.sigmoid(onset_pred) >= onset_th).float()
  offset_bin = (torch.sigmoid(offset_pred) >= offset_th).float()
  vel_bin = torch.round(vel_pred * 127)

  # Build new note event matrix
  new_note = update_note_event(prev_note[:,-1,:,:], onset_bin, offset_bin, vel_bin)

  # Store results
  results_onsets.append(onset_bin.squeeze(0).cpu())
  results_offsets.append(offset_bin.squeeze(0).cpu())
  results_notes.append(new_note.squeeze(0).cpu())

  # Prepare for next frame
  prev_note = new_note.clone()

frame_pred = reconstruct_frame_matrix(results_notes, vel=False)
vel_pred = reconstruct_frame_matrix(results_notes, vel=True)
onset_pred = reconstruct_onset_matrix(results_notes)
frame_pred = frame_pred[:Ts,:]
vel_pred = vel_pred[:Ts,:]
onset_pred = onset_pred[:Ts,:]

frame_acc = (frame_pred == frame_gt).float().mean()
print("Frame accuracy:", frame_acc.item())

onset_f1 = f1_score(onset_gt.flatten(), onset_pred.flatten())
print("Onset F1:", onset_f1)

# plot
plt.figure(figsize=(12,10))
plt.subplot(2,2,1)
plt.title("Predicted Velocity Matrix")
plt.imshow(vel_pred[:500,:].T, aspect='auto', origin='lower')
plt.colorbar()

plt.subplot(2,2,2)
plt.title("Ground Truth Velocity Matrix")
plt.imshow(vel_gt[:500,:].T, aspect='auto', origin='lower')
plt.colorbar()

plt.subplot(2,2,3)
plt.title("Predicted Onset Matrix")
plt.imshow(onset_pred[:500,:].T, aspect='auto', origin='lower')
plt.colorbar()

plt.subplot(2,2,4)
plt.title("Ground Truth Onset Matrix")
plt.imshow(onset_gt[:500,:].T, aspect='auto', origin='lower')
plt.colorbar()
plt.show()